# Combined Model — Feature Verification

## Objective

The objective of this notebook is to combine the selected feature representations from the four information groups and verify their consistency before model training.

Selected representations:

- G1: V1.1 — Overall student history
- G2: V2.4 — Fractional same-skill history
- G3: V3.2 — Fractional other-skill history
- G4: V4.3 — Question difficulty

The datasets were previously generated using five-fold cross-validation.

We will reuse the saved feature datasets without regenerating them.

This notebook performs feature loading, merging and verification only. Model training will be performed in a separate notebook.

In [1]:
import pandas as pd
from pathlib import Path

# Root directory of engineered features
feature_dir = Path(
    "../../../data/processed/engineered_features"
)

# Selected feature dataset directories
selected_feature_dirs = {
    "G1": feature_dir / "group_1_overall_history" / "v1.1",
    "G2": feature_dir / "group_2_same_skill_history" / "v2.4",
    "G3": feature_dir / "group_3_other_skill_history" / "v3.2",
    "G4": feature_dir / "group_4_question_difficulty" / "base_features"
}

# Display selected directories
for group, directory in selected_feature_dirs.items():
    print(f"{group}: {directory}")
    print("Directory exists:", directory.exists())
    print()

G1: ..\..\..\data\processed\engineered_features\group_1_overall_history\v1.1
Directory exists: True

G2: ..\..\..\data\processed\engineered_features\group_2_same_skill_history\v2.4
Directory exists: True

G3: ..\..\..\data\processed\engineered_features\group_3_other_skill_history\v3.2
Directory exists: True

G4: ..\..\..\data\processed\engineered_features\group_4_question_difficulty\base_features
Directory exists: True



In [2]:
# Store all feature datasets
fold_datasets = {}

for group, directory in selected_feature_dirs.items():

    fold_datasets[group] = {}

    for fold in range(1, 6):

        train_path = directory / f"fold_{fold}_train.csv"
        validation_path = directory / f"fold_{fold}_validation.csv"

        fold_datasets[group][fold] = {
            "train": pd.read_csv(train_path),
            "validation": pd.read_csv(validation_path)
        }

print("Successfully loaded all feature datasets.")

for group, folds in fold_datasets.items():

    print(f"\n{'='*50}")
    print(group)

    for fold, datasets in folds.items():

        train_df = datasets["train"]
        val_df = datasets["validation"]

        print(
            f"Fold {fold}: "
            f"Train {train_df.shape}, "
            f"Validation {val_df.shape}"
        )

Successfully loaded all feature datasets.

G1
Fold 1: Train (177324, 4), Validation (42659, 4)
Fold 2: Train (176273, 4), Validation (43710, 4)
Fold 3: Train (170186, 4), Validation (49797, 4)
Fold 4: Train (178311, 4), Validation (41672, 4)
Fold 5: Train (177838, 4), Validation (42145, 4)

G2
Fold 1: Train (177324, 5), Validation (42659, 5)
Fold 2: Train (176273, 5), Validation (43710, 5)
Fold 3: Train (170186, 5), Validation (49797, 5)
Fold 4: Train (178311, 5), Validation (41672, 5)
Fold 5: Train (177838, 5), Validation (42145, 5)

G3
Fold 1: Train (177324, 4), Validation (42659, 4)
Fold 2: Train (176273, 4), Validation (43710, 4)
Fold 3: Train (170186, 4), Validation (49797, 4)
Fold 4: Train (178311, 4), Validation (41672, 4)
Fold 5: Train (177838, 4), Validation (42145, 4)

G4
Fold 1: Train (177324, 13), Validation (42659, 13)
Fold 2: Train (176273, 13), Validation (43710, 13)
Fold 3: Train (170186, 13), Validation (49797, 13)
Fold 4: Train (178311, 13), Validation (41672, 13)
Fol

In [3]:
for group, folds in fold_datasets.items():

    print(f"\n{'='*60}")
    print(f"{group} — Columns")

    train_columns = folds[1]["train"].columns.tolist()
    val_columns = folds[1]["validation"].columns.tolist()

    print("\nTraining columns:")
    print(train_columns)

    print("\nValidation columns:")
    print(val_columns)


G1 — Columns

Training columns:
['user_id', 'order_id', 'correct', 'overall_smoothed_accuracy']

Validation columns:
['user_id', 'order_id', 'correct', 'overall_smoothed_accuracy']

G2 — Columns

Training columns:
['user_id', 'order_id', 'correct', 'weighted_same_skill_smoothed_accuracy', 'log_weighted_same_skill_previous_interactions']

Validation columns:
['user_id', 'order_id', 'correct', 'weighted_same_skill_smoothed_accuracy', 'log_weighted_same_skill_previous_interactions']

G3 — Columns

Training columns:
['user_id', 'order_id', 'correct', 'weighted_other_skill_smoothed_accuracy']

Validation columns:
['user_id', 'order_id', 'correct', 'weighted_other_skill_smoothed_accuracy']

G4 — Columns

Training columns:
['user_id', 'order_id', 'problem_id', 'correct', 'skill_id', 'skill_set', 'attempt_count', 'answer_type', 'question_previous_interactions', 'question_previous_correct', 'question_global_prior', 'question_smoothed_accuracy', 'log_question_previous_interactions']

Validation

In [4]:
# Features selected from each information group
feature_columns = {
    "G1": [
        "overall_smoothed_accuracy"
    ],

    "G2": [
        "weighted_same_skill_smoothed_accuracy",
        "log_weighted_same_skill_previous_interactions"
    ],

    "G3": [
        "weighted_other_skill_smoothed_accuracy"
    ],

    "G4": [
        "question_smoothed_accuracy",
        "log_question_previous_interactions",
        "answer_type"
    ]
}

# Columns used to identify each interaction
key_columns = ["user_id", "order_id"]
target_column = "correct"


def combine_feature_groups(group_datasets):

    # Start with G1 as the base dataset
    combined = group_datasets["G1"][
        key_columns + [target_column] + feature_columns["G1"]
    ].copy()

    # Check that the base dataset has unique interactions
    assert not combined.duplicated(key_columns).any()

    # Merge the remaining information groups
    for group in ["G2", "G3", "G4"]:

        group_df = group_datasets[group]

        # Retain only the selected features
        selected_df = group_df[
            key_columns + [target_column] + feature_columns[group]
        ].copy()

        # Check unique interaction identifiers
        assert not selected_df.duplicated(key_columns).any()

        # Verify that both datasets contain the same interactions
        base_index = pd.MultiIndex.from_frame(combined[key_columns])
        group_index = pd.MultiIndex.from_frame(selected_df[key_columns])

        assert base_index.isin(group_index).all()
        assert group_index.isin(base_index).all()

        # Verify that the actual target labels match
        group_labels = selected_df.set_index(key_columns)[target_column]
        aligned_labels = group_labels.reindex(
            pd.MultiIndex.from_frame(combined[key_columns])
        )

        assert aligned_labels.reset_index(drop=True).equals(
            combined[target_column].reset_index(drop=True)
        )

        # Merge only the selected feature columns
        combined = combined.merge(
            selected_df[key_columns + feature_columns[group]],
            on=key_columns,
            how="left",
            validate="one_to_one",
            sort=False
        )

    return combined

In [5]:
# Store combined training and validation datasets
combined_fold_datasets = {}

for fold in range(1, 6):

    group_train_datasets = {
        group: folds[fold]["train"]
        for group, folds in fold_datasets.items()
    }

    group_validation_datasets = {
        group: folds[fold]["validation"]
        for group, folds in fold_datasets.items()
    }

    combined_train = combine_feature_groups(
        group_train_datasets
    )

    combined_validation = combine_feature_groups(
        group_validation_datasets
    )

    combined_fold_datasets[fold] = {
        "train": combined_train,
        "validation": combined_validation
    }

    print(f"Fold {fold}")
    print("Training:", combined_train.shape)
    print("Validation:", combined_validation.shape)
    print()

Fold 1
Training: (177324, 10)
Validation: (42659, 10)

Fold 2
Training: (176273, 10)
Validation: (43710, 10)

Fold 3
Training: (170186, 10)
Validation: (49797, 10)

Fold 4
Training: (178311, 10)
Validation: (41672, 10)

Fold 5
Training: (177838, 10)
Validation: (42145, 10)



In [6]:
# Expected combined feature columns
expected_columns = (
    key_columns
    + [target_column]
    + feature_columns["G1"]
    + feature_columns["G2"]
    + feature_columns["G3"]
    + feature_columns["G4"]
)

for fold, datasets in combined_fold_datasets.items():

    print(f"\n{'='*60}")
    print(f"FOLD {fold}")

    for split in ["train", "validation"]:

        df = datasets[split]

        print(f"\n{split.upper()}")

        # Column verification
        assert df.columns.tolist() == expected_columns
        print("Correct columns:", True)

        # Missing values
        missing = df.isnull().sum().sum()
        print("Missing values:", missing)

        # Duplicate interactions
        duplicates = df.duplicated(
            subset=key_columns
        ).sum()
        print("Duplicate interactions:", duplicates)

        # Target distribution
        print("Target distribution:")
        print(df[target_column].value_counts(normalize=True).round(4))

        # Numerical feature ranges
        numeric_features = [
            column for column in expected_columns
            if column not in key_columns + [target_column, "answer_type"]
        ]

        print("\nNumerical feature ranges:")
        print(df[numeric_features].agg(["min", "max"]).round(4))

        # Answer type categories
        print("\nAnswer type categories:")
        print(df["answer_type"].value_counts().to_dict())

        # Assertions
        assert missing == 0
        assert duplicates == 0
        assert df[target_column].isin([0, 1]).all()
        assert df["answer_type"].notna().all()


FOLD 1

TRAIN
Correct columns: True
Missing values: 0
Duplicate interactions: 0
Target distribution:
correct
1    0.6601
0    0.3399
Name: proportion, dtype: float64

Numerical feature ranges:
     overall_smoothed_accuracy  weighted_same_skill_smoothed_accuracy  \
min                     0.0335                                    0.0   
max                     1.0000                                    1.0   

     log_weighted_same_skill_previous_interactions  \
min                                         0.0000   
max                                         5.0814   

     weighted_other_skill_smoothed_accuracy  question_smoothed_accuracy  \
min                                  0.0337                      0.1679   
max                                  1.0000                      1.0000   

     log_question_previous_interactions  
min                              0.0000  
max                              4.6821  

Answer type categories:
{'algebra': 146002, 'fill_in_1': 17826, 'choos

In [7]:
output_dir = Path(
    "../../../data/processed/engineered_features/combined_model"
)

output_dir.mkdir(parents=True, exist_ok=True)

for fold, datasets in combined_fold_datasets.items():

    for split, df in datasets.items():

        output_path = output_dir / f"fold_{fold}_{split}.csv"

        df.to_csv(output_path, index=False)

        print(f"Saved: {output_path}")

print("\nAll combined feature datasets saved successfully.")

Saved: ..\..\..\data\processed\engineered_features\combined_model\fold_1_train.csv
Saved: ..\..\..\data\processed\engineered_features\combined_model\fold_1_validation.csv
Saved: ..\..\..\data\processed\engineered_features\combined_model\fold_2_train.csv
Saved: ..\..\..\data\processed\engineered_features\combined_model\fold_2_validation.csv
Saved: ..\..\..\data\processed\engineered_features\combined_model\fold_3_train.csv
Saved: ..\..\..\data\processed\engineered_features\combined_model\fold_3_validation.csv
Saved: ..\..\..\data\processed\engineered_features\combined_model\fold_4_train.csv
Saved: ..\..\..\data\processed\engineered_features\combined_model\fold_4_validation.csv
Saved: ..\..\..\data\processed\engineered_features\combined_model\fold_5_train.csv
Saved: ..\..\..\data\processed\engineered_features\combined_model\fold_5_validation.csv

All combined feature datasets saved successfully.


In [8]:
output_dir = Path(
    "../../../data/processed/engineered_features/combined_model"
)

output_dir.mkdir(parents=True, exist_ok=True)

for fold, datasets in combined_fold_datasets.items():

    for split, df in datasets.items():

        output_path = output_dir / f"fold_{fold}_{split}.csv"

        df.to_csv(output_path, index=False)

        print(f"Saved: {output_path}")

print("\nAll combined feature datasets saved successfully.")

Saved: ..\..\..\data\processed\engineered_features\combined_model\fold_1_train.csv
Saved: ..\..\..\data\processed\engineered_features\combined_model\fold_1_validation.csv
Saved: ..\..\..\data\processed\engineered_features\combined_model\fold_2_train.csv
Saved: ..\..\..\data\processed\engineered_features\combined_model\fold_2_validation.csv
Saved: ..\..\..\data\processed\engineered_features\combined_model\fold_3_train.csv
Saved: ..\..\..\data\processed\engineered_features\combined_model\fold_3_validation.csv
Saved: ..\..\..\data\processed\engineered_features\combined_model\fold_4_train.csv
Saved: ..\..\..\data\processed\engineered_features\combined_model\fold_4_validation.csv
Saved: ..\..\..\data\processed\engineered_features\combined_model\fold_5_train.csv
Saved: ..\..\..\data\processed\engineered_features\combined_model\fold_5_validation.csv

All combined feature datasets saved successfully.
